# Model Training - EfficientNetB3

Este notebook entrena un modelo **EfficientNetB3** con transfer learning para la clasificación de defectos en acero.

**Objetivo de accuracy:** ≥92% (individual), ≥95% (ensemble)

**Arquitectura:**
- Base: EfficientNetB3 pre-entrenado en ImageNet
- Input: 224x224x3 (RGB)
- Data Augmentation: Rotation, Flip, Brightness, Contrast
- Class Weights para balance
- Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

TensorFlow version: 2.21.0
GPU available: []


In [2]:
# Configuration
BASE_DIR = Path(r'C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos')
DATA_DIR = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'ml_models' / 'efficientnet'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 4

CATEGORIES = ['cracks_scratches', 'flawless_prime', 'pitting_corrosion', 'surface_inclusions']

print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

Data directory: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\data\processed
Model directory: C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\ml_models\efficientnet
Image size: (224, 224)
Batch size: 32
Epochs: 50


## 1. Data Generators with Augmentation

In [3]:
# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    fill_mode='nearest'
)

# Validation and test generators (only rescaling)
val_test_datagen = ImageDataGenerator()

# Create generators
train_generator = train_datagen.flow_from_directory(
    DATA_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    DATA_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    DATA_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nTrain samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Test samples: {test_generator.samples}")
print(f"\nClass indices: {train_generator.class_indices}")

Found 1260 images belonging to 4 classes.
Found 270 images belonging to 4 classes.
Found 270 images belonging to 4 classes.

Train samples: 1260
Validation samples: 270
Test samples: 270

Class indices: {'cracks_scratches': 0, 'flawless_prime': 1, 'pitting_corrosion': 2, 'surface_inclusions': 3}


## 2. Compute Class Weights

In [4]:
# Calculate class weights to handle imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

print("Class weights:")
for idx, weight in class_weight_dict.items():
    class_name = list(train_generator.class_indices.keys())[list(train_generator.class_indices.values()).index(idx)]
    print(f"  {class_name}: {weight:.3f}")

Class weights:
  cracks_scratches: 0.750
  flawless_prime: 1.500
  pitting_corrosion: 1.500
  surface_inclusions: 0.750


## 3. Build EfficientNetB3 Model

In [5]:
def build_efficientnet_model(input_shape=(224, 224, 3), num_classes=4):
    """
    Build EfficientNetB3 model with transfer learning
    """
    # Load pre-trained EfficientNetB3
    base_model = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=input_shape)
    
    # Base model (EfficientNet includes internal Rescaling and Normalization)
    x = base_model(inputs, training=False)
    
    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    return model, base_model

# Build model
model, base_model = build_efficientnet_model(
    input_shape=(*IMG_SIZE, 3),
    num_classes=NUM_CLASSES
)

print(f"\nTotal parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print(f"Non-trainable parameters: {sum([tf.size(w).numpy() for w in model.non_trainable_weights]):,}")


Total parameters: 11,185,203
Trainable parameters: 398,084
Non-trainable parameters: 10,787,119


In [6]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

print("\nModel compiled successfully!")
model.summary()


Model compiled successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                        ┃ Output Shape               ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)          │ (None, 224, 224, 3)        │               0 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ efficientnetb3 (Functional)         │ (None, 7, 7, 1536)         │      10,783,535 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ global_average_pooling2d            │ (None, 1536)               │               0 │
│ (GlobalAveragePooling2D)            │                            │                 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ batch_normalization                 │ (None, 1536)               │           6,144 │
│ (BatchNormalization)                │                            │                 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout (Dropout)                   │ (None, 1536)               │               0 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense (Dense)                       │ (None, 256)                │         393,472 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ batch_normalization_1               │ (None, 256)                │           1,024 │
│ (BatchNormalization)                │                            │                 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                 │ (None, 256)                │               0 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                     │ (None, 4)                  │           1,028 │
└─────────────────────────────────────┴────────────────────────────┴─────────────────┘

 Total params: 11,185,203 (42.67 MB)

 Trainable params: 398,084 (1.52 MB)

 Non-trainable params: 10,787,119 (41.15 MB)

## 4. Setup Callbacks

In [7]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_DIR / 'efficientnet_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    TensorBoard(
        log_dir=str(MODEL_DIR / 'logs' / datetime.now().strftime('%Y%m%d-%H%M%S')),
        histogram_freq=0
    )
]

print("Callbacks configured:")
for cb in callbacks:
    print(f"  - {cb.__class__.__name__}")

Callbacks configured:
  - EarlyStopping
  - ReduceLROnPlateau
  - ModelCheckpoint
  - TensorBoard


## 5. Train Model - Phase 1 (Frozen Base)

In [8]:
print("="*80)
print("PHASE 1: Training with frozen base model")
print("="*80)

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 1 training completed!")

PHASE 1: Training with frozen base model
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2722 - auc: 0.5261 - loss: 2.0250 - precision: 0.2971 - recall: 0.1865
Epoch 1: val_accuracy improved from None to 0.16667, saving model to C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\ml_models\efficientnet\efficientnet_best.keras

Epoch 1: finished saving model to C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos\ml_models\efficientnet\efficientnet_best.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 86s 2s/step - accuracy: 0.2722 - auc: 0.5261 - loss: 2.0250 - precision: 0.2971 - recall: 0.1865 - val_accuracy: 0.1667 - val_auc: 0.5050 - val_loss: 1.4748 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2929 - auc: 0.5621 - loss: 1.6518 - precision: 0.3338 - recall: 0.1778
Epoch 2: val_accuracy did not improve from 0.16667
40/40 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.2929 - auc: 0.5621 - loss: 1.6

## 6. Fine-tuning - Phase 2 (Unfreeze Top Layers)

In [9]:
print("="*80)
print("PHASE 2: Fine-tuning - unfreezing top layers")
print("="*80)

# Unfreeze the top layers of the base model
base_model.trainable = True

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

print(f"Trainable layers: {sum([layer.trainable for layer in base_model.layers])}")
print(f"Non-trainable layers: {sum([not layer.trainable for layer in base_model.layers])}")

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

print(f"\nTotal trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print("Model recompiled with lower learning rate!")

PHASE 2: Fine-tuning - unfreezing top layers
Trainable layers: 20
Non-trainable layers: 365

Total trainable parameters: 3,771,236
Model recompiled with lower learning rate!


In [ ]:
# Continue training with fine-tuning
history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 2 fine-tuning completed!")

Epoch 1/30
20/40 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.2937 - auc: 0.5709 - loss: 1.4941 - precision: 0.3799 - recall: 0.1359

## 7. Evaluate Model on Test Set

In [ ]:
print("="*80)
print("EVALUATING MODEL ON TEST SET")
print("="*80)

# Load best model
best_model = keras.models.load_model(MODEL_DIR / 'efficientnet_best.keras')

# Evaluate
test_loss, test_acc, test_auc, test_precision, test_recall = best_model.evaluate(test_generator, verbose=1)

print(f"\n📊 TEST RESULTS:")
print(f"  Loss:      {test_loss:.4f}")
print(f"  Accuracy:  {test_acc*100:.2f}%")
print(f"  AUC:       {test_auc:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1-Score:  {2 * (test_precision * test_recall) / (test_precision + test_recall):.4f}")

## 8. Detailed Classification Report

In [ ]:
# Get predictions
test_generator.reset()
y_pred_probs = best_model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Classification report
class_names = list(test_generator.class_indices.keys())
print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## 9. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - EfficientNetB3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved!")

## 10. Training History Visualization

In [ ]:
# Combine histories
combined_history = {
    'accuracy': history_phase1.history['accuracy'] + history_phase2.history['accuracy'],
    'val_accuracy': history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy'],
    'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
    'val_loss': history_phase1.history['val_loss'] + history_phase2.history['val_loss']
}

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
axes[0].plot(combined_history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(combined_history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].axvline(x=len(history_phase1.history['accuracy']), color='red', linestyle='--', label='Fine-tuning starts')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(combined_history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(combined_history['val_loss'], label='Val Loss', linewidth=2)
axes[1].axvline(x=len(history_phase1.history['loss']), color='red', linestyle='--', label='Fine-tuning starts')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training history plot saved!")

## 11. Save Model and Metrics

In [ ]:
# Save final model
best_model.save(MODEL_DIR / 'efficientnet_final.keras')
print(f"✅ Final model saved to: {MODEL_DIR / 'efficientnet_final.keras'}")

# Save metrics
metrics = {
    'model_name': 'EfficientNetB3',
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'test_auc': float(test_auc),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'test_f1': float(2 * (test_precision * test_recall) / (test_precision + test_recall)),
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs_phase1': len(history_phase1.history['loss']),
    'epochs_phase2': len(history_phase2.history['loss']),
    'total_epochs': len(combined_history['loss']),
    'class_names': class_names,
    'training_date': datetime.now().isoformat()
}

with open(MODEL_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✅ Metrics saved to: {MODEL_DIR / 'metrics.json'}")

# Save class indices
with open(MODEL_DIR / 'class_indices.json', 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)

print(f"✅ Class indices saved to: {MODEL_DIR / 'class_indices.json'}")

## 12. Summary

In [ ]:
print("\n" + "="*80)
print("🎉 EFFICIENTNETB3 TRAINING COMPLETE")
print("="*80)
print(f"\n📊 FINAL RESULTS:")
print(f"  Test Accuracy:  {test_acc*100:.2f}%")
print(f"  Test AUC:       {test_auc:.4f}")
print(f"  Test F1-Score:  {metrics['test_f1']:.4f}")
print(f"\n💾 SAVED FILES:")
print(f"  • {MODEL_DIR / 'efficientnet_best.keras'}")
print(f"  • {MODEL_DIR / 'efficientnet_final.keras'}")
print(f"  • {MODEL_DIR / 'metrics.json'}")
print(f"  • {MODEL_DIR / 'class_indices.json'}")
print(f"  • {MODEL_DIR / 'confusion_matrix.png'}")
print(f"  • {MODEL_DIR / 'training_history.png'}")
print(f"\n📈 TARGET:")
if test_acc >= 0.92:
    print(f"  ✅ Target achieved! {test_acc*100:.2f}% ≥ 92%")
else:
    print(f"  ⚠️ Target not met: {test_acc*100:.2f}% < 92%")
    print(f"  Consider training longer or adjusting hyperparameters")
print("\n" + "="*80)